In [1]:
#!/usr/bin/env python3
"""One-cell Google Colab trainer for tydeptrai21042004/simulate-python.

No command-line arguments are read. Edit SETTINGS and run the whole file in one
normal Colab Python cell.
"""
from __future__ import annotations

import gc
import importlib.util
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import textwrap
import traceback
import zipfile

# ============================= SETTINGS =============================
REPO_URL = "https://github.com/tydeptrai21042004/simulate-python.git"
BRANCH = "main"
WORKDIR = "/content/simulate-python"
MODE = "official"          # "official" or "smoke"
REUSE_EXISTING_WORKDIR = True  # keep already-downloaded official sources in Colab
USE_GPU = True
RUN_TESTS = False
EXPORT_ONNX = False
PACKAGE = "/content/simulate_python_esp32_weights.zip"

# QUICK TRAINING SETTINGS
FAST_TRAIN = True
FAST_TRAIN_TIMESTAMPS = 3000
FAST_VAL_TIMESTAMPS = 500
FAST_TEST_TIMESTAMPS = 1000
FAST_BATCH_SIZE = 512
# ===================================================================


def run(cmd: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> None:
    print("\n+", " ".join(map(str, cmd)), flush=True)
    merged = os.environ.copy()
    if env:
        merged.update(env)
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, env=merged, check=True)


def clone_repo(repo_url: str, repo_dir: Path, branch: str) -> None:
    if REUSE_EXISTING_WORKDIR and (repo_dir / ".git").is_dir():
        print(f"[repo] reusing existing checkout: {repo_dir}", flush=True)
        print("[repo] existing downloaded data/source files will be preserved", flush=True)
        return
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", "--depth", "1", "--branch", branch, repo_url, str(repo_dir)])


def install_environment(repo_dir: Path) -> None:
    py = sys.executable
    run([py, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"])
    # Colab already ships a CUDA-matched PyTorch. Keep it instead of forcing a
    # second torch install; install the remaining explicit project dependencies.
    run([py, "-m", "pip", "install", "-q", "numpy>=1.26", "scipy>=1.11", "PyYAML>=6.0", "pytest>=8.0", "gdown>=5.2"])
    run([py, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=repo_dir)


def patch_cuda_training(repo_dir: Path) -> None:
    """Keep model on CUDA across validation epochs and add visible progress."""
    p = repo_dir / "src/uwb_tracking/esp32/training.py"
    text = p.read_text(encoding="utf-8")

    marker = "# COLAB_CUDA_FIX_V2"
    if marker not in text:
        old = (
            "    model.cpu()\n"
            "    return float(np.mean(np.concatenate(abs_errors))), float(np.mean(nlls)), float(np.mean(bces))\n"
        )
        new = (
            f"    {marker}: train_student keeps ownership of the model device\n"
            "    return float(np.mean(np.concatenate(abs_errors))), float(np.mean(nlls)), float(np.mean(bces))\n"
        )
        if old not in text:
            raise RuntimeError("Upstream evaluate_student() changed; CUDA patch no longer matches safely.")
        text = text.replace(old, new, 1)

    progress = "# COLAB_PROGRESS_V2"
    if progress not in text:
        old = "        score = val_mae\n        epochs_ran = epoch + 1\n"
        new = (
            "        score = val_mae\n"
            f"        # {progress}\n"
            "        print(\n"
            "            f\"[train] epoch={epoch + 1}/{epochs} val_mae_ns={val_mae:.6f} \"\n"
            "            f\"nll={val_nll:.6f} bce={val_bce:.6f}\", flush=True\n"
            "        )\n"
            "        epochs_ran = epoch + 1\n"
        )
        if old not in text:
            raise RuntimeError("Upstream train_student() changed; progress patch no longer matches safely.")
        text = text.replace(old, new, 1)

    p.write_text(text, encoding="utf-8")


def patch_official_data_nonfinite(repo_dir: Path) -> None:
    """Repair invalid official signal samples during MATLAB -> standard conversion."""
    p = repo_dir / "src/uwb_tracking/official_data.py"
    text = p.read_text(encoding="utf-8")
    marker = "# COLAB_OFFICIAL_NONFINITE_FIX_V3"
    if marker in text:
        return

    anchor = "def _get(raw: dict[str, object], candidates: list[str]) -> np.ndarray:\n    return np.asarray(raw[_find_key(raw, candidates)])\n"
    helper = '''def _get(raw: dict[str, object], candidates: list[str]) -> np.ndarray:
    return np.asarray(raw[_find_key(raw, candidates)])


# COLAB_OFFICIAL_NONFINITE_FIX_V3
def _repair_nonfinite_profiles(
    values: np.ndarray,
    name: str,
    *,
    nonnegative: bool = False,
) -> np.ndarray:
    """Repair NaN/Inf along the last (delay-bin) axis without dropping samples."""
    import warnings

    arr = np.asarray(values, dtype=np.float64).copy()
    if arr.ndim < 1 or arr.shape[-1] < 1:
        raise ValueError(f"{name}: empty profile array {arr.shape}")

    finite = np.isfinite(arr)
    bad_count = int(arr.size - np.count_nonzero(finite))
    if bad_count:
        flat = arr.reshape(-1, arr.shape[-1])
        x = np.arange(flat.shape[1], dtype=np.float64)
        finite_values = flat[np.isfinite(flat)]
        if finite_values.size == 0:
            raise ValueError(f"{name}: every sample is non-finite; cannot repair safely")

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            median_profile = np.nanmedian(
                np.where(np.isfinite(flat), flat, np.nan), axis=0
            )
        profile_good = np.isfinite(median_profile)
        if not np.any(profile_good):
            median_profile[:] = float(np.median(finite_values))
        elif not np.all(profile_good):
            median_profile[~profile_good] = np.interp(
                x[~profile_good], x[profile_good], median_profile[profile_good]
            )

        for row in flat:
            good = np.isfinite(row)
            if np.all(good):
                continue
            if not np.any(good):
                row[:] = median_profile
            elif np.count_nonzero(good) == 1:
                row[~good] = row[good][0]
            else:
                row[~good] = np.interp(x[~good], x[good], row[good])

        print(
            f"[data-fix] {name}: repaired {bad_count}/{arr.size} "
            f"non-finite values ({100.0 * bad_count / arr.size:.6f}%)",
            flush=True,
        )

    if nonnegative:
        np.maximum(arr, 0.0, out=arr)
    if not np.all(np.isfinite(arr)):
        raise ValueError(f"{name}: non-finite values remain after repair")
    return arr


def _repair_nonfinite_timeseries(
    values: np.ndarray,
    time: np.ndarray,
    name: str,
) -> np.ndarray:
    """Repair occasional NaN/Inf in time-major scalar/vector measurements."""
    arr = np.asarray(values, dtype=np.float64).copy()
    t = np.asarray(time, dtype=np.float64).reshape(-1)
    one_dimensional = arr.ndim == 1
    if one_dimensional:
        arr = arr[:, None]
    if arr.ndim != 2 or arr.shape[0] != t.size:
        raise ValueError(f"{name}: incompatible time-series shape {arr.shape}")
    if not np.all(np.isfinite(t)) or np.any(np.diff(t) <= 0):
        raise ValueError(f"{name}: source timestamps must be finite and increasing")

    bad_count = int(arr.size - np.count_nonzero(np.isfinite(arr)))
    if bad_count:
        for col in range(arr.shape[1]):
            y = arr[:, col]
            good = np.isfinite(y)
            if not np.any(good):
                raise ValueError(f"{name}: column {col} is entirely non-finite")
            if np.count_nonzero(good) == 1:
                y[~good] = y[good][0]
            elif not np.all(good):
                y[~good] = np.interp(t[~good], t[good], y[good])
        print(
            f"[data-fix] {name}: repaired {bad_count}/{arr.size} non-finite time-series values",
            flush=True,
        )
    if not np.all(np.isfinite(arr)):
        raise ValueError(f"{name}: non-finite values remain after repair")
    return arr[:, 0] if one_dimensional else arr
'''
    if anchor not in text:
        raise RuntimeError("Upstream official_data.py changed near _get(); finite-data patch cannot be applied safely.")
    text = text.replace(anchor, helper, 1)

    old = "        # This abs() is required for parity with the official MATLAB scripts.\n        cir = np.abs(cir_raw).astype(np.float64)\n"
    new = "        # This abs() is required for parity with the official MATLAB scripts.\n        cir = _repair_nonfinite_profiles(\n            np.abs(cir_raw), f\"Dyn_re_CIR{link}\"\n        )\n"
    if old not in text:
        raise RuntimeError("Upstream dynamic CIR conversion changed; finite-data patch cannot match safely.")
    text = text.replace(old, new, 1)

    old = "        ).astype(np.float64)\n        mu = _time_major(_get(raw, [f\"Dyn_re_MU{link}\"]), time.size, f\"Dyn_re_MU{link}\").astype(np.float64)\n"
    new = "        ).astype(np.float64)\n        var = _repair_nonfinite_profiles(\n            var, f\"Dyn_var_CIR{link}\", nonnegative=True\n        )\n        mu = _time_major(_get(raw, [f\"Dyn_re_MU{link}\"]), time.size, f\"Dyn_re_MU{link}\").astype(np.float64)\n        mu = _repair_nonfinite_timeseries(mu, time, f\"Dyn_re_MU{link}\")\n"
    if old not in text:
        raise RuntimeError("Upstream variance/MU conversion changed; finite-data patch cannot match safely.")
    text = text.replace(old, new, 1)

    old = "        if tof.size != time.size:\n            raise ValueError(f\"Dyn_real_ToF{link}: expected {time.size} values, got {tof.size}\")\n"
    new = "        if tof.size != time.size:\n            raise ValueError(f\"Dyn_real_ToF{link}: expected {time.size} values, got {tof.size}\")\n        tof = _repair_nonfinite_timeseries(tof, time, f\"Dyn_real_ToF{link}\")\n"
    if old not in text:
        raise RuntimeError("Upstream source ToF block changed; finite-data patch cannot match safely.")
    text = text.replace(old, new, 1)

    old = "        times.append(time)\n        cirs.append(cir)\n        variances.append(np.maximum(var, 0.0))\n        mus.append(mu[:, :2])\n        source_tofs.append(tof)\n        cir_bg.append(np.abs(bg_cir_raw).astype(np.float32))\n        var_bg.append(np.maximum(np.real(bg_var_raw), 0.0).astype(np.float32))\n        los.append(float(_get(raw, [f\"ToF_TRx{link}\"]).reshape(-1)[0]))\n"
    new = "        bg_cir = _repair_nonfinite_profiles(\n            np.abs(bg_cir_raw), f\"Bg_re_CIR{link}\"\n        )\n        bg_var = _repair_nonfinite_profiles(\n            np.real(bg_var_raw), f\"Bg_var_CIR{link}\", nonnegative=True\n        )\n        tof_los = float(_get(raw, [f\"ToF_TRx{link}\"]).reshape(-1)[0])\n        if not np.isfinite(tof_los):\n            raise ValueError(f\"ToF_TRx{link}: non-finite LOS ToF\")\n\n        times.append(time)\n        cirs.append(cir)\n        variances.append(var)\n        mus.append(mu[:, :2])\n        source_tofs.append(tof)\n        cir_bg.append(bg_cir.astype(np.float32))\n        var_bg.append(bg_var.astype(np.float32))\n        los.append(tof_los)\n"
    if old not in text:
        raise RuntimeError("Upstream background append block changed; finite-data patch cannot match safely.")
    text = text.replace(old, new, 1)

    old = "    anchors = _anchors_xy(_get(raw, [\"AnchorPos\", \"anchors\"]))\n    delay_grid = _get(raw, [\"re_SampTime\", \"delay_grid_ns\"]).reshape(-1).astype(np.float64)\n    if delay_grid.size < 1:\n        raise ValueError(\"re_SampTime/delay grid is empty\")\n"
    new = "    anchors = _anchors_xy(_get(raw, [\"AnchorPos\", \"anchors\"]))\n    if not np.all(np.isfinite(anchors)):\n        raise ValueError(\"AnchorPos contains non-finite coordinates\")\n    delay_grid = _get(raw, [\"re_SampTime\", \"delay_grid_ns\"]).reshape(-1).astype(np.float64)\n    if delay_grid.size < 1:\n        raise ValueError(\"re_SampTime/delay grid is empty\")\n    if not np.all(np.isfinite(delay_grid)) or (delay_grid.size > 1 and np.any(np.diff(delay_grid) <= 0)):\n        raise ValueError(\"re_SampTime/delay grid must be finite and strictly increasing\")\n"
    if old not in text:
        raise RuntimeError("Upstream structural data block changed; finite-data patch cannot match safely.")
    text = text.replace(old, new, 1)

    p.write_text(text, encoding="utf-8")

def patch_memory_safe_esp32_preprocessing(repo_dir: Path) -> None:
    """Avoid materializing unused paper-model tensors in ESP32-only official training.

    esp32s3_official.yaml disables teacher distillation, and train_esp32_pipeline.py
    consumes PreparedInputs.fusion rather than paper_cir/paper_var.  Keeping the
    original path when the env flag is absent means the rest of the repository
    and its tests retain their original behavior.
    """
    p = repo_dir / "src/uwb_tracking/data.py"
    text = p.read_text(encoding="utf-8")
    marker = "# COLAB_ESP32_FUSION_ONLY_V2"
    if marker in text:
        return

    if "import os\n" not in text:
        text = text.replace("from pathlib import Path\n", "from pathlib import Path\nimport os\n", 1)

    old = (
        "    paper_cir = np.stack([cir_dyn_n, cir_bg_n], axis=-1)[:, :, None, :, :]\n"
        "    paper_var = np.stack([var_dyn_n, var_bg_n], axis=-1)[:, :, None, :, :]\n"
        "    fusion = np.stack(\n"
        "        [cir_dyn_n, cir_bg_n, cir_diff_n, var_dyn_n, var_bg_n, var_diff_n],\n"
        "        axis=2,\n"
        "    )\n"
    )
    new = (
        f"    # {marker}\n"
        "    fusion_only = os.environ.get(\"UWB_ESP32_FUSION_ONLY\", \"0\") == \"1\"\n"
        "    if fusion_only:\n"
        "        paper_cir = None\n"
        "        paper_var = None\n"
        "    else:\n"
        "        paper_cir = np.stack([cir_dyn_n, cir_bg_n], axis=-1)[:, :, None, :, :]\n"
        "        paper_var = np.stack([var_dyn_n, var_bg_n], axis=-1)[:, :, None, :, :]\n"
        "    fusion = np.stack(\n"
        "        [cir_dyn_n, cir_bg_n, cir_diff_n, var_dyn_n, var_bg_n, var_diff_n],\n"
        "        axis=2,\n"
        "    )\n"
    )
    if old not in text:
        raise RuntimeError("Upstream prepare_inputs() changed; RAM patch no longer matches safely.")
    text = text.replace(old, new, 1)

    old_return = (
        "        paper_cir=paper_cir.reshape(n * l, 1, input_length, 2).astype(np.float32),\n"
        "        paper_var=paper_var.reshape(n * l, 1, input_length, 2).astype(np.float32),\n"
    )
    new_return = (
        "        paper_cir=(\n"
        "            np.empty((0, 1, input_length, 2), dtype=np.float32)\n"
        "            if paper_cir is None\n"
        "            else paper_cir.reshape(n * l, 1, input_length, 2).astype(np.float32)\n"
        "        ),\n"
        "        paper_var=(\n"
        "            np.empty((0, 1, input_length, 2), dtype=np.float32)\n"
        "            if paper_var is None\n"
        "            else paper_var.reshape(n * l, 1, input_length, 2).astype(np.float32)\n"
        "        ),\n"
    )
    if old_return not in text:
        raise RuntimeError("Upstream PreparedInputs return block changed; RAM patch no longer matches safely.")
    text = text.replace(old_return, new_return, 1)
    p.write_text(text, encoding="utf-8")


def patch_particle_filter_resampling(repo_dir: Path) -> None:
    """Fix float32 systematic-resampling edge case returning index == N."""
    p = repo_dir / "src/uwb_tracking/tracking/particle_filter.py"
    text = p.read_text(encoding="utf-8")
    marker = "# COLAB_SYSTEMATIC_RESAMPLE_FIX_V4"
    if marker in text:
        return

    old = '''def _systematic_resample(weights: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    n = weights.size
    positions = (rng.random() + np.arange(n)) / n
    cumulative = np.cumsum(weights)
    return np.searchsorted(cumulative, positions, side="left")
'''
    new = '''def _systematic_resample(weights: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    # COLAB_SYSTEMATIC_RESAMPLE_FIX_V4
    w = np.asarray(weights, dtype=np.float64).reshape(-1)
    n = w.size
    if n < 1:
        raise ValueError("weights must be non-empty")
    if not np.all(np.isfinite(w)) or np.any(w < 0):
        raise ValueError("weights must be finite and non-negative")
    total = float(np.sum(w, dtype=np.float64))
    if not np.isfinite(total) or total <= 0.0:
        raise ValueError("weights must have a positive finite sum")
    w /= total
    positions = (float(rng.random()) + np.arange(n, dtype=np.float64)) / float(n)
    cumulative = np.cumsum(w, dtype=np.float64)
    cumulative[-1] = 1.0
    idx = np.searchsorted(cumulative, positions, side="left")
    np.clip(idx, 0, n - 1, out=idx)
    return idx.astype(np.intp, copy=False)
'''
    if old not in text:
        raise RuntimeError("Upstream _systematic_resample() changed; PF patch cannot match safely.")
    p.write_text(text.replace(old, new, 1), encoding="utf-8")


def verify_particle_filter_patch(repo_dir: Path) -> None:
    """Regression-test the exact 128-particle failure before training."""
    src = str(repo_dir / "src")
    if src not in sys.path:
        sys.path.insert(0, src)
    import importlib
    import numpy as np
    import uwb_tracking.tracking.particle_filter as pf_module
    pf_module = importlib.reload(pf_module)
    _systematic_resample = pf_module._systematic_resample

    rng = np.random.default_rng(12345)
    w = np.full(128, np.float32(1.0 / 128.0), dtype=np.float32)
    w[-1] = np.nextafter(w[-1], np.float32(0.0))
    for _ in range(1000):
        idx = _systematic_resample(w, rng)
        if idx.size != 128 or int(idx.min()) < 0 or int(idx.max()) >= 128:
            raise RuntimeError(f"PF resampler regression failed: range={idx.min()}..{idx.max()}")
    print("[pf] systematic-resampling regression: OK (indices 0..127 only)", flush=True)


def patch_fast_dataset_caps(repo_dir: Path) -> None:
    """Use smaller subsets of the real official data for a quick Colab run."""
    if not FAST_TRAIN:
        return
    p = repo_dir / "scripts/train_esp32_pipeline.py"
    text = p.read_text(encoding="utf-8")
    marker = "# COLAB_FAST_REAL_DATA_CAPS_V4"
    if marker in text:
        return

    old = '''    train_core, val_idx = _split_train_val(
        train_idx, seed, float(cfg.get("validation_fraction", 0.15))
    )

    clean_obs = subset_observations(data, train_core)
'''
    new = f'''    train_core, val_idx = _split_train_val(
        train_idx, seed, float(cfg.get("validation_fraction", 0.15))
    )

    # {marker}: keep official data but cap timestamp counts for quick Colab training.
    fast_rng = np.random.default_rng(seed + 424242)
    fast_train_n = min({FAST_TRAIN_TIMESTAMPS}, int(train_core.size))
    if train_core.size > fast_train_n:
        train_core = np.sort(fast_rng.choice(train_core, size=fast_train_n, replace=False))
    fast_val_n = min({FAST_VAL_TIMESTAMPS}, int(val_idx.size))
    if val_idx.size > fast_val_n:
        start = max(0, (val_idx.size - fast_val_n) // 2)
        val_idx = val_idx[start:start + fast_val_n]
    fast_test_n = min({FAST_TEST_TIMESTAMPS}, int(test_idx.size))
    if test_idx.size > fast_test_n:
        test_idx = np.sort(test_idx)[:fast_test_n]
    print(
        f"[fast] timestamps: train={{train_core.size}} val={{val_idx.size}} test={{test_idx.size}}",
        flush=True,
    )

    clean_obs = subset_observations(data, train_core)
'''
    if old not in text:
        raise RuntimeError("Upstream train/val split block changed; fast-data patch cannot match safely.")
    p.write_text(text.replace(old, new, 1), encoding="utf-8")


def patch_pipeline_diagnostics(repo_dir: Path) -> None:
    """Add stage prints and release large training-only arrays before search/export."""
    p = repo_dir / "scripts/train_esp32_pipeline.py"
    text = p.read_text(encoding="utf-8")
    marker = "# COLAB_PIPELINE_DIAGNOSTICS_V2"
    if marker in text:
        return

    if "import gc\n" not in text:
        text = text.replace("import json\n", "import json\nimport gc\n", 1)

    text = text.replace(
        "    data = load_uwb_mat(data_path)\n",
        f"    print(\"[stage] loading standardized MAT dataset\", flush=True)\n    data = load_uwb_mat(data_path)\n    print(f\"[stage] dataset loaded: time={{data.num_time}} links={{data.num_links}} bins={{data.delay_grid_ns.size}}\", flush=True)\n    # {marker}\n",
        1,
    )
    text = text.replace(
        "    clean = prepare_inputs(data, clean_obs, input_length)\n",
        "    print(\"[stage] preprocessing train/validation tensors\", flush=True)\n    clean = prepare_inputs(data, clean_obs, input_length)\n",
        1,
    )
    text = text.replace(
        "    super_result = train_student(\n",
        "    print(f\"[stage] training supernet on {train_x.shape[0]} samples; val={val_x.shape[0]}\", flush=True)\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    super_result = train_student(\n",
        1,
    )
    p.write_text(text, encoding="utf-8")



def patch_pipeline_no_cli(repo_dir: Path) -> None:
    """Convert the cloned training entrypoint to a direct-call Colab entrypoint."""
    p = repo_dir / "scripts/train_esp32_pipeline.py"
    text = p.read_text(encoding="utf-8")
    marker = "# COLAB_NO_CLI_V2"
    if marker in text:
        return

    text = text.replace("import argparse\n", "", 1)
    old = """    parser = argparse.ArgumentParser(
        description="Train a progressive structured-LTH ESP32 student and export the selected ticket."
    )
    parser.add_argument("--config", default="configs/esp32s3.yaml")
    parser.add_argument("--no-teacher", action="store_true", help="Disable teacher distillation.")
    parser.add_argument("--auto-data", action="store_true", help="Auto-fetch official data if data_path is missing.")
    parser.add_argument("--onnx", action="store_true", help="Also export ONNX opset 18.")
    parser.add_argument("--espdl", action="store_true", help="Also quantize/export .espdl with ESP-PPQ.")
    args = parser.parse_args()

    cfg = yaml.safe_load((ROOT / args.config).read_text(encoding="utf-8"))
"""
    new = f"""    # {marker}: fixed Colab runtime config; no argument parser.
    config_path = "configs/colab_runtime.yaml"
    auto_data = True
    no_teacher = False
    export_onnx_cli = {bool(EXPORT_ONNX)!r}
    export_espdl_cli = False

    cfg = yaml.safe_load((ROOT / config_path).read_text(encoding="utf-8"))
"""
    if old not in text:
        raise RuntimeError("Upstream pipeline CLI block changed; no-CLI patch no longer matches safely.")
    text = text.replace(old, new, 1)
    text = text.replace("_resolve_data(cfg, args.auto_data)", "_resolve_data(cfg, auto_data)")
    text = text.replace("and not args.no_teacher", "and not no_teacher")
    text = text.replace("bool(args.onnx or export_cfg.get(\"onnx\", False))", "bool(export_onnx_cli or export_cfg.get(\"onnx\", False))")
    text = text.replace("bool(args.espdl or export_cfg.get(\"espdl\", False))", "bool(export_espdl_cli or export_cfg.get(\"espdl\", False))")
    if "args." in text or "argparse" in text:
        raise RuntimeError("No-CLI patch incomplete: argument parsing tokens remain in cloned pipeline.")
    p.write_text(text, encoding="utf-8")

def make_runtime_config(repo_dir: Path, selected_device: str) -> tuple[Path, str]:
    import torch
    import yaml

    base = "configs/esp32s3_official.yaml" if MODE == "official" else "configs/esp32s3_smoke.yaml"
    cfg = yaml.safe_load((repo_dir / base).read_text(encoding="utf-8"))
    device = selected_device
    cfg["output_dir"] = "results/colab_esp32s3_official" if MODE == "official" else "results/colab_esp32s3_smoke"
    training_cfg = cfg.setdefault("training", {})
    training_cfg["device"] = device
    # Quick real-data run: 10 epochs per required training stage.
    training_cfg["epochs_supernet"] = 10
    training_cfg["epochs_ticket"] = 10
    training_cfg["epochs_control"] = 10
    training_cfg["batch_size"] = FAST_BATCH_SIZE
    training_cfg["patience"] = 3
    cfg["use_teacher_distillation"] = False

    if FAST_TRAIN:
        # Train ONE structured LTH candidate instead of all five.
        cfg["ticket_candidates"] = [{"channels": [8, 12, 12], "hidden": 16}]
        # For a short artifact-producing run, export the best ticket even when
        # the full paper/research quality guard is not met in only 10 epochs.
        lth = cfg.setdefault("lth_search", {})
        lth["require_accuracy_guard"] = False
        lth["tracking_guard_particles"] = 64

        deploy = cfg.setdefault("deployment_evaluation", {})
        deploy["enabled"] = True
        deploy["scenarios"] = ["los"]
        deploy.setdefault("particle_filter", {})["num_particles"] = 64
        cfg.setdefault("export", {})["calibration_samples"] = 256

    print(
        f"[runtime] quick training: epochs=10/10/10 batch={training_cfg['batch_size']} "
        f"ticket_candidates={len(cfg.get('ticket_candidates', []))}",
        flush=True,
    )
    cfg.setdefault("export", {})["onnx"] = bool(EXPORT_ONNX)
    if MODE == "official":
        cfg.setdefault("official_data", {})["auto_download"] = True
    out = repo_dir / "configs/colab_runtime.yaml"
    out.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    return out, device


def cuda_preflight() -> str:
    import torch
    if not USE_GPU or not torch.cuda.is_available():
        print("[cuda] CUDA not selected/available; using CPU", flush=True)
        return "cpu"
    try:
        dev = torch.device("cuda")
        x = torch.randn(8, 6, 176, device=dev)
        net = torch.nn.Conv1d(6, 8, 7, stride=2, padding=3).to(dev)
        y = net(x).square().mean()
        y.backward()
        torch.cuda.synchronize()
        del x, net, y
        torch.cuda.empty_cache()
        print("[cuda] preflight Conv1D forward/backward: OK", flush=True)
        print("[cuda] GPU:", torch.cuda.get_device_name(0), flush=True)
        return "cuda"
    except Exception as exc:
        print("[cuda] GPU preflight failed; falling back to CPU:", repr(exc), flush=True)
        return "cpu"


def verify(repo_dir: Path) -> None:
    run([sys.executable, "-m", "compileall", "-q", "src", "scripts", "tests"], cwd=repo_dir)
    if RUN_TESTS:
        run([sys.executable, "-m", "pytest", "-q"], cwd=repo_dir, env={"PYTHONPATH": str(repo_dir / "src")})


def prepare_data(repo_dir: Path) -> None:
    if MODE != "official":
        p = repo_dir / "data/uwb_demo_input.mat"
        if not p.exists() or p.stat().st_size < 1024:
            raise RuntimeError(f"Dataset is missing/incomplete: {p}")
        print(f"[data] ready: {p} ({p.stat().st_size / 2**20:.1f} MiB)", flush=True)
        return

    import numpy as np
    from scipy.io import loadmat

    p = repo_dir / "data/uwb_original_standard.mat"

    def audit(path: Path) -> bool:
        if not path.exists() or path.stat().st_size < 1024:
            return False
        try:
            bg = loadmat(
                path, squeeze_me=True,
                variable_names=["cir_background", "var_background"],
            )
            ok = True
            for name in ("cir_background", "var_background"):
                arr = np.asarray(bg[name])
                bad = int(arr.size - np.count_nonzero(np.isfinite(arr)))
                print(f"[data-audit] {name}: shape={arr.shape} nonfinite={bad}", flush=True)
                ok = ok and bad == 0
            if np.any(np.asarray(bg["var_background"]) < 0):
                print("[data-audit] var_background contains negative values", flush=True)
                ok = False
            del bg
            gc.collect()
            return bool(ok)
        except Exception as exc:
            print(f"[data-audit] existing standardized MAT unusable: {exc}", flush=True)
            return False

    if audit(p):
        print("[data] reusing already-valid standardized official MAT; no reconversion", flush=True)
    else:
        print("[data] standardized MAT absent/invalid; rebuilding once with finite-value repair", flush=True)
        run([sys.executable, "scripts/fetch_original_data.py", "--force-convert"], cwd=repo_dir)
        if not audit(p):
            raise ValueError("Official standardized MAT is still invalid after repaired conversion")

    print(f"[data] ready: {p} ({p.stat().st_size / 2**20:.1f} MiB)", flush=True)



def run_pipeline_in_process(repo_dir: Path) -> None:
    """Run patched sources directly so Colab shows the real traceback."""
    src = str(repo_dir / "src")
    if src not in sys.path:
        sys.path.insert(0, src)

    # A previous failed Colab run may have old repo modules cached in memory.
    # Purge them so this run definitely executes the just-patched Python files.
    for module_name in list(sys.modules):
        if module_name == "uwb_tracking" or module_name.startswith("uwb_tracking."):
            del sys.modules[module_name]

    os.environ["PYTHONUNBUFFERED"] = "1"
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    # Safe because the ESP32 official config disables teacher distillation.
    os.environ["UWB_ESP32_FUSION_ONLY"] = "1"

    module_path = repo_dir / "scripts/train_esp32_pipeline.py"
    spec = importlib.util.spec_from_file_location("colab_train_esp32_pipeline", module_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot import {module_path}")
    module = importlib.util.module_from_spec(spec)
    try:
        spec.loader.exec_module(module)
        module.main()
    finally:
        os.environ.pop("UWB_ESP32_FUSION_ONLY", None)

def summarize_and_package(repo_dir: Path, output_dir: str, package_path: Path) -> None:
    out = repo_dir / output_dir
    report_path = out / "pipeline_report.json"
    if not report_path.exists():
        raise FileNotFoundError(f"Training finished without pipeline report: {report_path}")
    report = json.loads(report_path.read_text(encoding="utf-8"))
    if report.get("status") != "ok":
        raise RuntimeError(f"Pipeline status: {report.get('status')}")

    required = [
        out / "checkpoints/best_student.pt",
        out / "export/ufuse_weights_int8.bin",
        out / "export/ufuse_weights_int8.h",
        out / "export/ufuse_weights_manifest.json",
        out / "export/golden_vectors.npz",
        out / "export/uwb_background_u8.bin",
        out / "export/uwb_runtime_constants.h",
        out / "export/uwb_runtime_manifest.json",
        out / "export/export_report.json",
        out / "pipeline_report.json",
        out / "deployment_tracking_results.json",
        out / "deployment_tracking_results.csv",
    ]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing expected artifacts:\n" + "\n".join(missing))

    package_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(package_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in required:
            zf.write(p, arcname=p.relative_to(out))
        prov = repo_dir / "data/uwb_original_standard.mat.provenance.json"
        if prov.exists():
            zf.write(prov, arcname="data_provenance/uwb_original_standard.mat.provenance.json")
        zf.write(repo_dir / "configs/colab_runtime.yaml", arcname="config/colab_runtime.yaml")

    sel = report["selected_for_deployment"]
    exp = report["export"]
    print("\n=== FINAL MODEL ===")
    print("type:", sel["type"])
    print("arch:", sel["arch"])
    print("parameters:", sel["parameters"])
    print("checkpoint:", out / "checkpoints/best_student.pt")
    print("INT8 weights:", out / "export/ufuse_weights_int8.bin")
    print("weight bytes:", exp["weight_blob_bytes"])
    print("ZIP:", package_path)


def show_failure_context(repo_dir: Path) -> None:
    print("\n================ REAL FAILURE CONTEXT ================", file=sys.stderr)
    print("The exception above is the actual pipeline error (not a child-process wrapper error).", file=sys.stderr)
    for rel in (
        "results/colab_esp32s3_official/pipeline_report.json",
        "results/colab_esp32s3_smoke/pipeline_report.json",
    ):
        p = repo_dir / rel
        if p.exists():
            print(f"\nPartial report: {p}", file=sys.stderr)
            try:
                print(p.read_text(encoding="utf-8")[-6000:], file=sys.stderr)
            except Exception:
                pass


def main() -> None:
    if MODE not in {"official", "smoke"}:
        raise ValueError("MODE must be 'official' or 'smoke'")

    repo_dir = Path(WORKDIR).resolve()
    output_dir = "results/colab_esp32s3_official" if MODE == "official" else "results/colab_esp32s3_smoke"
    print(textwrap.dedent(f"""
        ============================================================
        simulate-python : QUICK Colab trainer V4
        ============================================================
        repo:       {REPO_URL}
        branch:     {BRANCH}
        mode:       {MODE}
        workdir:    {repo_dir}
        use GPU:    {USE_GPU}
        run tests:  {RUN_TESTS}
        reuse repo: {REUSE_EXISTING_WORKDIR}
        final ZIP:  {PACKAGE}
        ============================================================
    """).strip())

    clone_repo(REPO_URL, repo_dir, BRANCH)
    install_environment(repo_dir)
    patch_cuda_training(repo_dir)
    patch_official_data_nonfinite(repo_dir)
    patch_memory_safe_esp32_preprocessing(repo_dir)
    patch_particle_filter_resampling(repo_dir)
    patch_fast_dataset_caps(repo_dir)
    patch_pipeline_diagnostics(repo_dir)
    patch_pipeline_no_cli(repo_dir)
    verify_particle_filter_patch(repo_dir)

    selected_device = cuda_preflight()
    config, device = make_runtime_config(repo_dir, selected_device)
    print("[runtime] training device:", device, flush=True)

    # Tests run with fusion-only mode disabled, preserving normal repository behavior.
    verify(repo_dir)
    prepare_data(repo_dir)

    try:
        run_pipeline_in_process(repo_dir)
    except Exception:
        traceback.print_exc()
        show_failure_context(repo_dir)
        raise

    summarize_and_package(repo_dir, output_dir, Path(PACKAGE).resolve())
    print("\nSUCCESS - training and export completed.")
    try:
        from google.colab import files
        files.download(PACKAGE)
    except Exception:
        pass


main()


simulate-python : QUICK Colab trainer V4
repo:       https://github.com/tydeptrai21042004/simulate-python.git
branch:     main
mode:       official
workdir:    /content/simulate-python
use GPU:    True
run tests:  False
reuse repo: True
final ZIP:  /content/simulate_python_esp32_weights.zip

+ git clone --depth 1 --branch main https://github.com/tydeptrai21042004/simulate-python.git /content/simulate-python

+ /usr/bin/python3 -m pip install -q --upgrade pip setuptools wheel

+ /usr/bin/python3 -m pip install -q numpy>=1.26 scipy>=1.11 PyYAML>=6.0 pytest>=8.0 gdown>=5.2

+ /usr/bin/python3 -m pip install -q -e . --no-deps
[pf] systematic-resampling regression: OK (indices 0..127 only)
[cuda] preflight Conv1D forward/backward: OK
[cuda] GPU: Tesla T4
[runtime] quick training: epochs=10/10/10 batch=512 ticket_candidates=1
[runtime] training device: cuda

+ /usr/bin/python3 -m compileall -q src scripts tests
[data] standardized MAT absent/invalid; rebuilding once with finite-value repair


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>